In [57]:
from __future__ import print_function, division
import os, random, time, copy
import numpy as np
import pandas as pd
import os.path as path
import scipy.io as sio
from scipy import misc
from scipy import ndimage, signal
import scipy
import pickle
import sys
import math
import matplotlib.pyplot as plt
import PIL.Image
from io import BytesIO


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler 
import torch.nn.functional as F
from torch.autograd import Variable
import torchvision
from torchvision import datasets, models, transforms
import torchvision.utils as vutils

import warnings 
warnings.filterwarnings("ignore")
print(sys.version)
print(torch.__version__)


manualSeed = 999
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)

3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
2.5.1+cu121
Random Seed:  999


In [58]:
torch.manual_seed(0)

exp_dir = './res1030'

modelFlag = 'Res18sc'

project_name = 'ukm1030_GANfea_v1_' + modelFlag

device ='cpu'
if torch.cuda.is_available(): 
    device='cuda:0'


total_epoch_num = 50 
batch_size = 16   
insertConv = False    
embDimension = 64
isPretrained = False

newsize = (64, 64)

path_to_feats = './feats' 
pklName = path.join(path_to_feats, modelFlag.lower()+'.pkl')

nc = 3
nz = 100
ngf = 64
ndf = 64
beta1 = 0.5
ngpu = 1




nClassTotal = 200
nClassCloseset = nClassTotal


lr = 0.0001 

num_epochs = total_epoch_num
torch.cuda.device_count()
torch.cuda.empty_cache()

save_dir="./res1030/dcgan2nslmal"
print(save_dir)    
if not os.path.exists(save_dir): os.makedirs(save_dir)

log_filename = os.path.join(save_dir, 'train.log')

./res1030/dcgan2nslmal


In [59]:
class Generatorzy(nn.Module):
    def __init__(self, z_dim):
        super(Generatorzy, self).__init__()
        
        self.fc = nn.Linear(z_dim, 256*8*8)
        self.g_deconv_1 = nn.Sequential(
                          nn.ConvTranspose2d(256, 128, kernel_size=3,
                                    stride= 2, padding=(3-2+1)//2,
                                    output_padding = (3-2)%2), 
                          nn.BatchNorm2d(128),
                          nn.LeakyReLU()
                          )
        self.g_deconv_2 = nn.Sequential(
                          nn.ConvTranspose2d(128, 64, kernel_size=3,
                                    stride= 1, padding=(3-1+1)//2,
                                    output_padding = (3-1)%2), 
                          nn.BatchNorm2d(64),
                          nn.LeakyReLU()
                          )
        self.g_deconv_3 = nn.Sequential(
                          nn.ConvTranspose2d(64, 3, kernel_size=3,
                                    stride= 2, padding=(3-2+1)//2,
                                    output_padding = (3-2)%2),
                          nn.Tanh()
                          )
        self.fczy = nn.Linear(3*32*32, 100)  
        

    def forward(self, x):
        x = self.fc(x).view(-1, 256, 8, 8)
        x = self.g_deconv_1(x)
        x = self.g_deconv_2(x)
        x = self.g_deconv_3(x)
        x_zy = self.fczy(x.view(-1,3*32*32))
        return x_zy,x    


    
class Discriminatorzy(nn.Module):
    def __init__(self):
        super(Discriminatorzy, self).__init__()
        
        self.d_conv_1 = nn.Sequential(
                          nn.Conv2d(3, 32, kernel_size=3,
                                    stride=2, padding=1), 
                          nn.LeakyReLU()
                          )
        self.d_conv_2 = nn.Sequential(
                          nn.Conv2d(32, 64, kernel_size=3,
                                    stride=2, padding=1), 
                          nn.LeakyReLU()
                          )
        self.d_conv_3 = nn.Sequential(
                          nn.Conv2d(64, 128, kernel_size=3,
                                    stride=2, padding=0), 
                          nn.LeakyReLU()
                          )
        self.fc = nn.Linear(3*3*128, 1)
        self.fczy = nn.Linear(3*3*128, 46)

    def forward(self, x):
        x = self.d_conv_1(x)
        x = self.d_conv_2(x)
        x = self.d_conv_3(x)
        x = x.view(-1, 128*3*3)
        x_zy = self.fczy(x)
        x = torch.sigmoid(self.fc(x))
        return x_zy,x

In [60]:
print(device)

bestEpoch =2 
z_dim=100

discriminator = Discriminatorzy()
netDzy = discriminator.to(device)
generator = Generatorzy(z_dim)
netGzy = generator.to(device)


path_to_G = os.path.join(save_dir, 'dcgan-2nsl-epoch-{}.GNet'.format(bestEpoch))
path_to_D = os.path.join(save_dir, 'dcgan-2nsl-epoch-{}.DNet'.format(bestEpoch))
netGzy.load_state_dict(torch.load(path_to_G))
netDzy.load_state_dict(torch.load(path_to_D))



noise = torch.randn(batch_size, z_dim, device=device)#64--->1 64---->1

_,fake = netGzy(noise)
_,predLabel = netDzy(fake)

print(noise.shape, fake.shape, predLabel.shape)

cuda:0
torch.Size([16, 100]) torch.Size([16, 3, 32, 32]) torch.Size([16, 1])


In [61]:
featestsc=[]

for i in range(10000):
    noise = torch.randn(1, z_dim, device=device)
    features1,predConf = netGzy(noise)
    features2,yuzhi = netDzy(predConf.detach())
    #print("yuzhi:",yuzhi)
    features1=features1.cpu().detach().numpy().tolist()
    if yuzhi>=0.5017:
        print("yuzhi:",yuzhi)
        featestsc.append(features1)
    

featestsc = np.array(featestsc)
featestsc=torch.squeeze(torch.tensor(featestsc)).numpy()
print("donesc0815")

yuzhi: tensor([[0.5018]], device='cuda:0', grad_fn=<SigmoidBackward0>)
yuzhi: tensor([[0.5018]], device='cuda:0', grad_fn=<SigmoidBackward0>)
yuzhi: tensor([[0.5018]], device='cuda:0', grad_fn=<SigmoidBackward0>)
yuzhi: tensor([[0.5019]], device='cuda:0', grad_fn=<SigmoidBackward0>)
yuzhi: tensor([[0.5017]], device='cuda:0', grad_fn=<SigmoidBackward0>)
yuzhi: tensor([[0.5018]], device='cuda:0', grad_fn=<SigmoidBackward0>)
yuzhi: tensor([[0.5018]], device='cuda:0', grad_fn=<SigmoidBackward0>)
donesc0815


In [62]:
predConf.shape

torch.Size([1, 3, 32, 32])

In [63]:
np.save("./data/sCkitsune-augment1023malD05.npy",featestsc)
print("featestsc",featestsc.shape)

featestsc (7, 100)
